# Freeze rare-class metadata

Run after 10, once per frozen split. Very few samples means fewer than 50 original images. Test support below 10 is recorded separately. These flags are class metadata. Original image labels stay intact.

In [ ]:
import csv
from collections import Counter
from hashlib import sha256
from pathlib import Path

root = Path("E:/VLM")
data_dir = root / "data/hyperkvasir_23_v1"
manifest_path = data_dir / "split_manifest.csv"
rare_path = data_dir / "rare_labels.csv"
manifest_digest = sha256(manifest_path.read_bytes()).hexdigest()
rare_cutoff = 50

if not rare_path.exists():
    with manifest_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    original = Counter(row["label"] for row in rows)
    counts = Counter((row["label"], row["split"]) for row in rows)
    fresh = Counter((row["label"], row["split"]) for row in rows if not row["prior_use"])
    temporary = rare_path.with_suffix(".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(("label", "original_count", "train", "validation", "test", "fresh_test",
                         "very_few_samples", "test_support_below_10", "split_sha256"))
        for label in sorted(original):
            writer.writerow((label, original[label], counts[label, "train"],
                             counts[label, "validation"], counts[label, "test"], fresh[label, "test"],
                             int(original[label] < rare_cutoff), int(counts[label, "test"] < 10),
                             manifest_digest))
    temporary.replace(rare_path)

with rare_path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 23 and all(row["split_sha256"] == manifest_digest for row in rows)
print("Label | Source | Train | Validation | Test | Fresh test")
for row in rows:
    if row["very_few_samples"] == "1" or row["test_support_below_10"] == "1":
        print(*(row[key] for key in ("label", "original_count", "train", "validation",
                                    "test", "fresh_test")), sep=" | ")